In [ ]:
import os, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection

try:
    from sqlalchemy import create_engine
    import psycopg2
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sqlalchemy", "psycopg2-binary"])
    from sqlalchemy import create_engine

USER = os.getenv("POSTGRES_USER", "projet")
PWD  = os.getenv("POSTGRES_PASSWORD", "projet")
DB   = os.getenv("POSTGRES_DB", "enedis")
HOST = os.getenv("POSTGRES_HOST", "postgres")

engine = create_engine(f"postgresql+psycopg2://{USER}:{PWD}@{HOST}:5432/{DB}")

df_jour   = pd.read_sql("SELECT * FROM gold_conso_jour_region", engine)
df_thermo = pd.read_sql("SELECT * FROM gold_thermosensibilite_region ORDER BY rang_reactivite", engine)
df_heure  = pd.read_sql("SELECT * FROM gold_pointe_horaire", engine)
df_froids = pd.read_sql("SELECT * FROM gold_jours_froids", engine)

df_jour["date"] = pd.to_datetime(df_jour["date"])
df_froids["date"] = pd.to_datetime(df_froids["date"])

DOCS = "../docs"
os.makedirs(DOCS, exist_ok=True)

plt.rcParams.update({"figure.facecolor": "white", "axes.spines.top": False, "axes.spines.right": False})
print(f"Tables chargées : jour={len(df_jour):,}, thermo={len(df_thermo)}, heure={len(df_heure):,}, froids={len(df_froids)}")

In [ ]:
nat = df_jour.groupby("date", as_index=False).agg(
    puissance_gw=("puissance_moy_mw", lambda s: s.sum() / 1000),
    tmoy_c=("tmoy_c", "mean"),
)
nat["mm7"] = nat["puissance_gw"].rolling(7, center=True).mean()

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(nat["date"], nat["puissance_gw"], color="#CBD5E1", linewidth=1, label="Puissance moyenne journalière")
ax.plot(nat["date"], nat["mm7"], color="#1D4ED8", linewidth=2.2, label="Moyenne mobile 7 jours")

# Mise en évidence de la vague de froid du 12–17 décembre 2022
ax.axvspan(pd.Timestamp("2022-12-11"), pd.Timestamp("2022-12-18"), color="#FEE2E2", alpha=0.8)
pic = nat.loc[nat["puissance_gw"].idxmax()]
ax.annotate(
    f"Vague de froid déc. 2022\n{pic['puissance_gw']:.1f} GW ({pic['tmoy_c']:.1f} °C)",
    xy=(pic["date"], pic["puissance_gw"]),
    xytext=(pd.Timestamp("2022-08-20"), pic["puissance_gw"] - 2),
    arrowprops=dict(arrowstyle="->", color="#B91C1C"),
    fontsize=9.5, fontweight="bold", color="#991B1B",
)

ax.set_title("La consommation ≤ 36 kVA double presque entre l'été et la vague de froid de décembre 2022",
             fontsize=12, fontweight="bold", loc="left", pad=12)
ax.set_ylabel("Puissance moyenne (GW)")
ax.set_ylim(0, nat["puissance_gw"].max() * 1.12)
ax.legend(frameon=False, loc="upper center")
ax.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
fig.savefig(f"{DOCS}/05_evolution_annuelle.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
prof = (df_heure[df_heure["mois"].isin([7, 12])]
        .groupby(["mois", "famille_profil", "heure"], as_index=False)["puissance_moy_mw"].sum())
prof["puissance_gw"] = prof["puissance_moy_mw"] / 1000

couleurs = {"RES": "#DC2626", "PRO": "#2563EB", "ENT": "#64748B"}
noms = {"RES": "Résidentiel (RES)", "PRO": "Professionnels (PRO)", "ENT": "Entreprises (ENT)"}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, mois, titre in [(axes[0], 12, "Hiver (Décembre)"), (axes[1], 7, "Été (Juillet)")]:
    sub = prof[prof["mois"] == mois]
    for fam in ["RES", "PRO", "ENT"]:
        d = sub[sub["famille_profil"] == fam]
        ax.plot(d["heure"], d["puissance_gw"], label=noms[fam],
                color=couleurs[fam], linewidth=2.5 if fam == "RES" else 1.6)
    ax.set_title(titre, fontsize=11, fontweight="bold")
    ax.set_xlabel("Heure de la journée (heure de Paris)")
    ax.set_xticks(range(0, 24, 3))
    ax.grid(axis="y", linestyle="--", alpha=0.3)

axes[0].axvline(19, color="#DC2626", linestyle=":", alpha=0.7)
axes[0].annotate("Pointe 19 h\n(tirée par le RES)", xy=(19, 25), xytext=(12, 26),
                 arrowprops=dict(arrowstyle="->", color="#DC2626"), color="#991B1B", fontsize=9, fontweight="bold")
axes[0].set_ylabel("Puissance moyenne nationale (GW)")
axes[0].set_ylim(0, None)
axes[1].legend(frameon=False, loc="upper right")

fig.suptitle("En hiver, la pointe de 19 h est massivement portée par les clients résidentiels",
             fontsize=12.5, fontweight="bold", y=1.02)
plt.tight_layout()
fig.savefig(f"{DOCS}/05_profil_horaire_hiver_ete.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={"width_ratios": [1, 1.3]})

# --- AVANT : camembert naïf, palette arc-en-ciel, titre neutre ---
conso_reg = df_jour.groupby("nom_region")["energie_mwh"].sum().sort_index()
couleurs_naive = plt.cm.hsv(np.linspace(0, 0.9, len(conso_reg)))
ax1.pie(conso_reg, labels=conso_reg.index, colors=couleurs_naive, autopct="%1.0f%%", textprops={"fontsize": 8})
ax1.set_title("AVANT — Consommation", fontsize=11, color="#B91C1C", fontweight="bold")

# --- APRÈS : barres horizontales triées, accent pré-attentif sur le top 3, axe à 0 ---
t = df_thermo.sort_values("gradient_w_par_client", ascending=True)
bar_colors = ["#DC2626" if r <= 3 else "#94A3B8" for r in t["rang_reactivite"]]
bars = ax2.barh(t["nom_region"], t["gradient_w_par_client"], color=bar_colors)
for bar, val in zip(bars, t["gradient_w_par_client"]):
    ax2.text(val + 0.8, bar.get_y() + bar.get_height() / 2, f"{val:.1f} W/°C",
             va="center", fontsize=8.5, fontweight="bold" if val >= 46.9 else "normal")

ax2.set_xlim(0, t["gradient_w_par_client"].max() * 1.18)
ax2.set_xlabel("Gradient de thermosensibilité par client (W / °C perdu, sous 15 °C)")
ax2.set_title("APRÈS — Bretagne, Normandie et Pays de la Loire réagissent le plus par client",
              fontsize=11, color="#15803D", fontweight="bold", loc="left")
ax2.grid(axis="x", linestyle="--", alpha=0.3)

plt.tight_layout()
fig.savefig(f"{DOCS}/05_avant_apres_camembert_barres.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
t_mw = df_thermo.sort_values("gradient_mw_par_degre", ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- AVANT : axe démarrant à 50, ordre alphabétique, barres verticales illisibles ---
t_alpha = df_thermo.sort_values("nom_region")
ax1.bar(t_alpha["nom_region"], t_alpha["gradient_mw_par_degre"], color="#6366F1")
ax1.set_ylim(50, 220)
ax1.tick_params(axis="x", rotation=75, labelsize=8)
ax1.set_ylabel("Gradient")
ax1.set_title("AVANT — Axe tronqué à 50 (amplifie faussement les écarts)",
              fontsize=10.5, color="#B91C1C", fontweight="bold")

# --- APRÈS : axe à 0, trié, horizontal, unités explicites ---
t_mw_asc = t_mw.sort_values("gradient_mw_par_degre", ascending=True)
colors = ["#1D4ED8" if v >= 160 else "#94A3B8" for v in t_mw_asc["gradient_mw_par_degre"]]
bars = ax2.barh(t_mw_asc["nom_region"], t_mw_asc["gradient_mw_par_degre"], color=colors)
for bar, val in zip(bars, t_mw_asc["gradient_mw_par_degre"]):
    ax2.text(val + 3, bar.get_y() + bar.get_height() / 2, f"{val:.1f} MW/°C", va="center", fontsize=8.5)

ax2.set_xlim(0, t_mw_asc["gradient_mw_par_degre"].max() * 1.2)
ax2.set_xlabel("Gradient régional total (MW supplémentaires / °C perdu, sous 15 °C)")
ax2.set_title("APRÈS — Axe à 0 : l'Île-de-France (+208,5 MW/°C) pèse le plus en volume brut",
              fontsize=10.5, color="#15803D", fontweight="bold", loc="left")
ax2.grid(axis="x", linestyle="--", alpha=0.3)

plt.tight_layout()
fig.savefig(f"{DOCS}/05_avant_apres_axe_zero.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
idf = df_jour[df_jour["nom_region"] == "Île-de-France"].sort_values("date")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- AVANT : double axe trompeur ---
ax1_bis = ax1.twinx()
ax1.plot(idf["date"], idf["puissance_moy_mw"], color="tab:blue", label="Puissance (MW)")
ax1_bis.plot(idf["date"], idf["tmoy_c"], color="tab:red", alpha=0.7, label="Température (°C)")
ax1.set_ylabel("Puissance (MW)", color="tab:blue")
ax1_bis.set_ylabel("Température (°C)", color="tab:red")
ax1.set_title("AVANT — Double axe : les croisements sont arbitraires",
              fontsize=10.5, color="#B91C1C", fontweight="bold")
ax1.tick_params(axis="x", rotation=30)

# --- APRÈS : nuage de points + seuil 15 °C + droite de régression ---
sous_15 = idf[idf["tmoy_c"] < 15]
sur_15  = idf[idf["tmoy_c"] >= 15]

ax2.scatter(sur_15["tmoy_c"], sur_15["puissance_moy_mw"], color="#94A3B8", alpha=0.55, s=22, label="≥ 15 °C (hors chauffage)")
ax2.scatter(sous_15["tmoy_c"], sous_15["puissance_moy_mw"], color="#1D4ED8", alpha=0.65, s=22, label="< 15 °C (régime chauffage)")

pente, intercept = np.polyfit(sous_15["tmoy_c"], sous_15["puissance_moy_mw"], 1)
xs = np.linspace(sous_15["tmoy_c"].min(), 15, 50)
ax2.plot(xs, pente * xs + intercept, color="#DC2626", linewidth=2.2,
         label=f"Pente : {-pente:.1f} MW / °C perdu")
ax2.axvline(15, color="#475569", linestyle="--", linewidth=1)
ax2.text(15.3, idf["puissance_moy_mw"].max() * 0.95, "Seuil 15 °C", fontsize=8.5, color="#475569")

ax2.set_xlabel("Température moyenne journalière (°C)")
ax2.set_ylabel("Puissance moyenne journalière (MW)")
ax2.set_title("APRÈS — Sous 15 °C, chaque degré perdu ajoute 208,5 MW en Île-de-France",
              fontsize=10.5, color="#15803D", fontweight="bold", loc="left")
ax2.legend(frameon=False, fontsize=8.5)
ax2.grid(linestyle="--", alpha=0.25)

plt.tight_layout()
fig.savefig(f"{DOCS}/05_avant_apres_double_axe_nuage.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fichiers_geo = glob.glob("../data/**/*.geojson", recursive=True)
if not fichiers_geo:
    print("Aucun fichier .geojson trouvé dans ../data/ : vérifie le dossier data/raw/.")
else:
    with open(fichiers_geo[0], encoding="utf-8") as f:
        geo = json.load(f)

    valeurs = dict(zip(df_thermo["code_region"].astype(str), df_thermo["gradient_w_par_client"]))
    patches, vals = [], []
    for feat in geo["features"]:
        code = str(feat["properties"].get("code", ""))
        if code not in valeurs:
            continue
        geom = feat["geometry"]
        polys = [geom["coordinates"]] if geom["type"] == "Polygon" else geom["coordinates"]
        for poly in polys:
            patches.append(Polygon(poly[0], closed=True))
            vals.append(valeurs[code])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
    for ax, cmap, titre, couleur_titre in [
        (ax1, "jet", "AVANT — Palette arc-en-ciel (non ordonnée, illisible daltoniens)", "#B91C1C"),
        (ax2, "YlOrRd", "APRÈS — Palette séquentielle : la façade Ouest est la plus thermosensible", "#15803D"),
    ]:
        pc = PatchCollection(patches, cmap=cmap, edgecolor="white", linewidth=0.8)
        pc.set_array(np.array(vals))
        ax.add_collection(pc)
        ax.autoscale_view()
        ax.set_aspect(1.4)
        ax.axis("off")
        ax.set_title(titre, fontsize=10, color=couleur_titre, fontweight="bold")
        cbar = fig.colorbar(pc, ax=ax, fraction=0.035, pad=0.02)
        cbar.set_label("W / °C par client")

    plt.tight_layout()
    fig.savefig(f"{DOCS}/05_avant_apres_carte.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
## Synthèse des corrections avant / après (principes de perception)

| Avant (version naïve) | Après (version corrigée) | Principe de perception appliqué |
|---|---|---|
| Camembert de la consommation par région | Barres horizontales triées (gradient W/°C/client) | **Hiérarchie de Cleveland & McGill** : la position sur un axe commun se compare beaucoup plus précisément qu'un angle ou une aire. |
| Barres verticales dont l'axe démarre à 50 | Barres horizontales triées, axe démarrant à 0 | **Intégrité visuelle (lie factor)** : la longueur d'une barre doit être proportionnelle à la valeur représentée. |
| Double axe consommation / température dans le temps | Nuage de points température vs puissance + droite sous 15 °C | **Éviter les doubles axes** : les croisements dépendent de l'échelle choisie ; un nuage montre directement la relation et le seuil de 15 °C. |
| Carte en palette arc-en-ciel (`jet`) | Carte en palette séquentielle (`YlOrRd`) | **Perception de la couleur** : une grandeur ordonnée appelle un dégradé monotone, lisible en niveaux de gris et par les daltoniens. |
| Titre descriptif neutre (« Consommation »), sans unités | Titre conclusif orienté décision, unités explicites, top 3 en couleur d'accent | **Attributs pré-attentifs & ratio données/encre (Tufte)** : le lecteur identifie immédiatement les régions prioritaires. |